# 82-0 Game Logic Walkthrough

This notebook explains the scoring engine behind the 82-0 NBA draft game and walks through
the strategy logic implemented in this project.

Player data is fetched live from the game's public Firebase URL — the same request any browser makes when loading the game. No data files are bundled in this repo.

In [ ]:
import sys, json, urllib.request
from pathlib import Path
from collections import Counter, defaultdict

sys.path.insert(0, str(Path('.').resolve()))

from engine import (
    calculate_team_result, projected_wins, adjust_spg_bpg,
    ERA_CEILINGS, POSITIONS, TEAM_GRADES
)
from simulator import load_players, player_slug, can_player_play_position
from optimizer import stat_contribution, build_top_players_by_pos, build_market_context

print('Setup complete.')

---
## 1. Fetching Player Data

The game loads its player roster from a public Firebase Storage URL with no authentication.
We fetch it the same way the game does.

In [ ]:
DATA_URL = (
    'https://firebasestorage.googleapis.com/v0/b/'
    'project-4599904239656435772.firebasestorage.app/o/'
    'players_flat.json?alt=media'
)

with urllib.request.urlopen(DATA_URL) as resp:
    raw_players = json.load(resp)

print(f'Total records fetched: {len(raw_players):,}')
print(f'Sample record: {raw_players[0]}')

era_counts = Counter(p['era'] for p in raw_players if p.get('era'))
print('\nRecords per era:')
for era in sorted(era_counts):
    print(f'  {era}: {era_counts[era]:,}')

In [ ]:
# Build the simulator's indexed data structures from the fetched data
import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump(raw_players, f)
    tmp_path = f.name

teams, valid_combos, all_players = load_players(tmp_path)
ctx = build_market_context(teams, valid_combos)
top_by_pos = build_top_players_by_pos(all_players)

print(f'Players after processing: {len(all_players):,}')
print(f'Valid (team, era) combos: {len(valid_combos)}')
print(f'Teams: {len(teams)}')

---
## 2. The Spin Mechanic

Each draft round picks an era uniformly (1/7 chance each), then picks a team uniformly within that era.
Because early eras have fewer teams, each specific old-era combo is more likely than a modern one.

In [ ]:
era_team_count = Counter(era for _, era in valid_combos)

print(f"{'Era':<8} {'Teams':>6} {'P(era)':>8} {'P(one combo)':>14} {'vs uniform 1/180':>18}")
print('-' * 58)
for era in ['1960s','1970s','1980s','1990s','2000s','2010s','2020s']:
    n = era_team_count[era]
    p_combo = (1/7) / n
    p_uniform = 1 / len(valid_combos)
    ratio = p_combo / p_uniform
    print(f"{era:<8} {n:>6} {1/7:>8.4f} {p_combo:>14.5f} {ratio:>17.2f}x")

print()
print('GSW 1960s (Wilt) probability: 1/7 × 1/14 = 1/98 ≈ 1.02%')
print('vs uniform 1/180 ≈ 0.56% — era-first makes elite 1960s combos nearly 2x more likely')

---
## 3. The SPG/BPG Adjustment

1960s players have no recorded steals or blocks. The engine scales up whoever *does* have
those stats to compensate — treating the recorded players as representatives for the whole team.

```
adj_SPG = sum(spg where spg > 0)  ×  (5 / count_with_valid_spg)
adj_BPG = sum(bpg where bpg > 0)  ×  (5 / count_with_valid_bpg)
```

If nobody has valid stats, the contribution is simply 0.

In [ ]:
modern = {'ppg': 20, 'rpg': 8, 'apg': 3, 'spg': 2.0, 'bpg': 1.5}
null_era = {'ppg': 30, 'rpg': 18, 'apg': 4, 'spg': None, 'bpg': None}

cases = [
    ('5 modern players',           [modern] * 5),
    ('1 old-era + 4 modern',       [null_era] + [modern] * 4),
    ('2 old-era + 3 modern',       [null_era] * 2 + [modern] * 3),
    ('5 old-era (all null)',        [null_era] * 5),
]

print(f"{'Case':<28} {'Valid SPG':>10} {'Scale':>7} {'adj_SPG':>9} {'adj_BPG':>9}")
print('-' * 65)
for label, roster in cases:
    adj = adjust_spg_bpg(roster)
    n_valid = sum(1 for p in roster if p.get('spg') and p['spg'] > 0)
    scale = f'{5/n_valid:.2f}x' if n_valid > 0 else 'n/a'
    print(f"{label:<28} {n_valid:>10} {scale:>7} {adj['adjustedSpg']:>9.2f} {adj['adjustedBpg']:>9.2f}")

print()
print('Takeaway: pairing a 1960s star with a high-SPG/BPG modern player amplifies their defensive stats.')

---
## 4. Team OVR and Win Projection

Team OVR is a weighted sum of five cumulative stats against benchmark totals for a full team:

```
OVR = 100 × (
    Σ PPG / 133.4 × 0.46  +   # scoring  (46%)
    Σ RPG / 39.7  × 0.25  +   # rebounds (25%)
    Σ APG / 29.3  × 0.18  +   # assists  (18%)
    adj_SPG / 6.1 × 0.07  +   # steals    (7%)
    adj_BPG / 3.2 × 0.04       # blocks    (4%)
)

Wins = round(82 × min(OVR / 110, 1) ^ 1.15)
```

82-0 requires OVR ≥ ~110.

In [ ]:
# Win curve — shows the non-linear relationship
print(f"{'OVR':>5}  {'Wins':>5}  {'Bar'}")
print('-' * 40)
for ovr in range(60, 116, 5):
    wins = projected_wins(ovr)
    bar = '#' * (wins // 2)
    marker = ' ← 82-0 threshold' if wins == 82 and ovr <= 112 else ''
    print(f"{ovr:>5}  {wins:>5}  {bar}{marker}")

In [ ]:
# Real lineup example
example_roster = [
    {'player': 'Wilt Chamberlain', 'ppg': 41.46, 'rpg': 25.1,  'apg': 3.02, 'spg': None, 'bpg': None},
    {'player': 'Michael Jordan',   'ppg': 32.65, 'rpg': 6.15,  'apg': 5.93, 'spg': 2.81, 'bpg': 1.18},
    {'player': 'Giannis',          'ppg': 29.6,  'rpg': 11.3,  'apg': 6.0,  'spg': 1.0,  'bpg': 1.1},
    {'player': 'Oscar Robertson',  'ppg': 29.66, 'rpg': 8.73,  'apg': 10.5, 'spg': None, 'bpg': None},
    {'player': 'Bob McAdoo',       'ppg': 28.24, 'rpg': 12.67, 'apg': 2.59, 'spg': 1.14, 'bpg': 2.42},
]

result = calculate_team_result(example_roster)
print('Wilt + MJ + Giannis + Oscar + McAdoo:')
print(f'  Sum PPG: {sum(p["ppg"] for p in example_roster):.2f}')
print(f'  Sum RPG: {sum(p["rpg"] for p in example_roster):.2f}')
print(f'  Sum APG: {sum(p["apg"] for p in example_roster):.2f}')
adj = adjust_spg_bpg(example_roster)
print(f'  adj_SPG: {adj["adjustedSpg"]:.2f}  (3 of 5 players have valid SPG, scale = 5/3)')
print(f'  adj_BPG: {adj["adjustedBpg"]:.2f}')
print(f'  Team OVR: {result["teamOvr"]}')
print(f'  Wins: {result["wins"]}  Grade: {result["grade"]} — {result["label"]}')

---
## 5. Era Ceilings

The individual player rating (HoopIQ mode only) normalises stats against era-specific peaks.
Classic mode uses raw absolute values — which is why Wilt's 41 PPG season is irreplaceable.

In [ ]:
print(f"{'Era':<8} {'PPG ceil':>10} {'RPG ceil':>10} {'APG ceil':>10} {'SPG ceil':>10} {'BPG ceil':>10}")
print('-' * 58)
for era in ['1960s','1970s','1980s','1990s','2000s','2010s','2020s']:
    c = ERA_CEILINGS[era]
    print(f"{era:<8} {c['ppg']:>10} {c['rpg']:>10} {c['apg']:>10} {c['spg']:>10} {c['bpg']:>10}")

print()
wilt_ppg = 41.46
print(f'Wilt 1960s: {wilt_ppg} PPG vs ceiling of {ERA_CEILINGS["1960s"]["ppg"]} = {wilt_ppg/ERA_CEILINGS["1960s"]["ppg"]*100:.0f}% of era peak')
print(f'In classic mode this counts at face value — Wilt contributes {wilt_ppg/133.4*0.46*100:.1f}% of the benchmark PPG target alone')

---
## 6. Player Ranking — The Mental Score

The optimizer ranks players by `stat_contribution` — their direct weighted contribution to team OVR.
For human use, this maps to a simpler mental formula:

```
Mental Score = PPG + 2×RPG + 2×APG + 3×SPG + 3×BPG
```

Thresholds: **S ≥ 80 / A ≥ 63 / B ≥ 50 / reroll < 46**

In [ ]:
def mental(p):
    return (
        (p.get('ppg') or 0) +
        2 * (p.get('rpg') or 0) +
        2 * (p.get('apg') or 0) +
        3 * (p.get('spg') or 0) +
        3 * (p.get('bpg') or 0)
    )

def tier(ms):
    if ms >= 80: return 'S'
    if ms >= 63: return 'A'
    if ms >= 50: return 'B'
    if ms >= 46: return 'C'
    return 'Reroll'

# Top 30 unique players
seen = {}
for p in sorted(all_players, key=mental, reverse=True):
    if p['player'] not in seen:
        seen[p['player']] = p
    if len(seen) >= 30:
        break

print(f"{'#':>3}  {'Tier':>5}  {'Score':>6}  {'Player':<30} {'Team':<5} {'Era':<8} {'PPG':>6} {'RPG':>6} {'APG':>6} {'SPG':>6} {'BPG':>6}")
print('-' * 95)
for i, p in enumerate(sorted(seen.values(), key=mental, reverse=True), 1):
    ms = mental(p)
    spg_str = f"{p.get('spg'):.1f}" if p.get('spg') else 'null'
    bpg_str = f"{p.get('bpg'):.1f}" if p.get('bpg') else 'null'
    name = p['player'][:28]
    print(f"{i:>3}  {tier(ms):>5}  {ms:>6.1f}  {name:<30} {p['team']:<5} {p['decade']:<8} {p.get('ppg') or 0:>6.1f} {p.get('rpg') or 0:>6.1f} {p.get('apg') or 0:>6.1f} {spg_str:>6} {bpg_str:>6}")

---
## 7. Best Target Per Era

When you land on an era, these are the teams worth targeting — and when to use your era reroll.

In [ ]:
for era in ['1960s','1970s','1980s','1990s','2000s','2010s','2020s']:
    combos = [
        (team, ctx['combo_players'][(team, era)])
        for team in ctx['era_to_teams'].get(era, ())
        if ctx['combo_players'].get((team, era))
    ]
    combos.sort(key=lambda x: mental(x[1][0]), reverse=True)
    print(f'{era}  ({len(combos)} teams):')
    for team, players in combos[:5]:
        best = players[0]
        ms = mental(best)
        t = tier(ms)
        name = best['player'][:25]
        print(f'  [{t}] {ms:5.1f}  {team:4s}  {name}')
    print()

---
## 8. The 82-0 Threshold

What combined mental score do 5 valid players need to reach 82 wins?

In [ ]:
import itertools

# Deduplicate top 35 by name, keep highest mental score version
best_version = {}
for p in sorted(all_players, key=mental, reverse=True):
    name = p['player']
    if name not in best_version:
        best_version[name] = p
    if len(best_version) >= 35:
        break
top35 = list(best_version.values())

min_82 = None
for combo in itertools.combinations(top35, 5):
    assigned = {}
    for pos in POSITIONS:
        for p in combo:
            if p not in assigned.values() and can_player_play_position(p, pos):
                assigned[pos] = p
                break
    if len(assigned) < 5:
        continue
    result = calculate_team_result(list(assigned.values()))
    total_ms = sum(mental(p) for p in assigned.values())
    if result['wins'] >= 82:
        if min_82 is None or total_ms < min_82[0]:
            min_82 = (total_ms, result, dict(assigned))

total_ms, result, assigned = min_82
print(f'Minimum mental score total for 82-0: {total_ms:.1f}')
print(f'OVR={result["teamOvr"]}  wins={result["wins"]}')
print()
for pos, p in assigned.items():
    print(f'  {pos}: {p["player"]:28s}  {p["team"]} {p["decade"]}  mental={mental(p):.1f}')

print()
print('Safe target: combined mental score ≥ 320')
print('  APG-heavy lineups (playmakers) need ~325 — assists are slightly overweighted in the mental formula')
print('  BPG-heavy lineups (rim protectors) can clear 82-0 at ~315 — blocks are underweighted')

---
## 9. Reroll Decision Logic

The optimizer precomputes the **expected best player contribution per slot** across all possible spins.
This is the baseline: if your current spin's best pick is below this, a reroll is positive EV.

In [ ]:
print('Expected baseline contribution per slot (era-first weighted average):')
print()
print(f"  {'Slot':<5} {'sc baseline':>12}  {'mental equiv':>13}  {'reroll if best pick mental score <'}")
print('  ' + '-' * 70)
for pos in POSITIONS:
    baseline_sc = ctx['expected_pos_baseline'][pos]
    baseline_mental = baseline_sc * 290
    reroll_threshold = round(baseline_mental / 5) * 5  # round to nearest 5
    print(f"  {pos:<5} {baseline_sc:>12.4f}  {baseline_mental:>13.1f}  ~{reroll_threshold}")

print()
print('The optimizer uses team reroll when: best available pick sc < era average')
print('The optimizer uses era reroll when: best available pick sc < team-across-eras average')
print()
print('Human shortcut: reroll if best player mental score < 50')

---
## 10. Quick Strategy Reference

In [ ]:
print('=' * 55)
print('  QUICK STRATEGY REFERENCE')
print('=' * 55)
print()
print('MENTAL SCORE = PPG + 2×RPG + 2×APG + 3×SPG + 3×BPG')
print('  (skip SPG/BPG for 1960s players — not recorded)')
print()
print('TIERS:')
print('  S  ≥ 80   Lock in, never question')
print('  A  ≥ 63   Strong pick')
print('  B  ≥ 50   Take if rerolls gone')
print('  C  ≥ 46   Marginal, last resort')
print('       < 46   Reroll or restart')
print()
print('ROUND 1: Restart if best available < 63 (Tier A)')
print()
print('REROLLS:')
print('  Team reroll  → same era, new team (use if best pick < 50)')
print('  Era reroll   → same team, new era (use to target a legend)')
print()
print('TARGET COMBOS (era reroll):')
targets = [
    ('GSW','1960s','Wilt',97.7), ('PHI','1960s','Wilt',89.1),
    ('MIL','1970s','Kareem',83.6), ('LAL','1970s','Kareem',78.1),
    ('DEN','2020s','Jokic',76.9), ('WAS','2020s','Westbrook',74.0),
    ('DAL','2020s','Luka',70.7),  ('MIL','2020s','Giannis',70.5),
    ('HOU','1990s','Hakeem',68.9),('CHI','1980s','MJ',68.8),
    ('SAS','1990s','Robinson',68.2),('SAC','1960s','Oscar',68.1),
]
for team, era, player, ms in targets:
    print(f'  {team} → {era}: {player} ({ms})')
print()
print('82-0 TARGET: combined mental score ≥ 320')
print('  Need 1 Tier S  OR  2-3 Tier A players, no dead weight')